<a href="https://colab.research.google.com/github/SaharaAli16/UNT_Teaching/blob/main/DTSC_4050/Week_7_Distributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTSC 4050 — Lesson 5 Programming Activity
# Probability and Distributions - part II

# Learning goals
By the end of this activity, you should be able to:

1. Distinguish a probability distribution from an empirical distribution.
2. Use simulation to investigate the law of averages.
3. Draw random samples with and without replacement.
4. Distinguish a population parameter from a sample statistic.
5. Simulate the empirical distribution of a statistic.
6. Explain how sample size affects sampling variability.




## 1. Setup

Run this cell first. We will use NumPy, pandas, and Matplotlib, consistent with the previous programming activity.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# random number generator
rng = np.random.default_rng()

pd.set_option("display.max_rows", 20)


# Part A — Probability Distribution vs. Empirical Distribution

For a fair six-sided die, the theoretical probability of each face is:

P(1)= P(2)= ... = P(6) = 1/6

That theoretical distribution exists even before we roll the die.

An empirical distribution is based on what we actually observe in a sample or simulation.


In [ ]:
faces = np.arange(1, 7)
theoretical_probability = np.repeat(1/6, 6)

probability_distribution = pd.DataFrame({
    "Face": faces,
    "Probability": theoretical_probability
})

probability_distribution


In [ ]:
#theoretic distribution of truly random dice
plt.bar(faces, theoretical_probability)
plt.xticks(faces)
plt.ylim(0, 0.45)
plt.xlabel("Die face")
plt.ylabel("Probability")
plt.title("Theoretical Probability Distribution of a Fair Die")
plt.show()


# Activity 1 — Simulate 10 rolls

Do you expect every face to appear exactly 1/6 of the time in only 10 rolls? Why or why not?


In [ ]:
# generate 10 die rolls.

rolls_10 = rng.choice(faces, size=10)

print("Rolls:", rolls_10)

# count the observed proportion of each face
empirical_10 = np.array([(rolls_10 == face).mean() for face in faces])

results_10 = pd.DataFrame({
    "Face": faces,
    "Theoretical": theoretical_probability,
    "Empirical (10 rolls)": empirical_10
})

results_10


In [ ]:
# run this cell after re-running the cell above.

plt.bar(faces, empirical_10)
plt.axhline(1/6, linestyle="--", label="Theoretical probability = 1/6")
plt.xticks(faces)
plt.ylim(0, 0.45)
plt.xlabel("Die face")
plt.ylabel("Observed proportion")
plt.title("Empirical Distribution — 10 Rolls")
plt.legend()
plt.show()


### Quick Check 1

1. Does the empirical distribution for 10 rolls exactly match the theoretical distribution?
2. Re-run the simulation three times. What changes?
3. Which distribution is theoretical, and which is observed?


# Part B — The Law of Averages

The law of averages says that when a chance experiment is repeated independently under the same conditions, the observed proportion of an event tends to get closer to its theoretical probability as the number of repetitions increases.

For a fair die, the long-run proportion of rolling a 4 should approach \(1/6\).


### Activity 2 — Small vs. large samples

Complete the function, then compare 10, 100, 1,000, and 100,000 rolls.


In [ ]:
def empirical_die_distribution(n):
    # simulate n rolls
    rolls = rng.choice(faces, size=n)

    # calculate the proportion for each face
    proportions = np.array([(rolls == face).mean() for face in faces])
    return proportions

sample_sizes = [10, 100, 1_000, 100_000]

comparison = pd.DataFrame({"Face": faces})

for n in sample_sizes:
    comparison[f"n={n:,}"] = empirical_die_distribution(n)

comparison["Theoretical"] = 1/6
comparison


In [ ]:
# visual comparison
for n in sample_sizes:
    proportions = empirical_die_distribution(n)
    plt.figure(figsize=(6, 3))
    plt.bar(faces, proportions)
    plt.axhline(1/6, linestyle="--", label="Theoretical = 1/6")
    plt.xticks(faces)
    plt.ylim(0, 0.45)
    plt.xlabel("Die face")
    plt.ylabel("Observed proportion")
    plt.title(f"Empirical Distribution — {n:,} Rolls")
    plt.legend()
    plt.show()


### Quick Check 2

1. Which sample size usually looks most like the theoretical distribution?
2. Does the law of averages say that a short run must be balanced?
3. Suppose you roll five 6s in a row. Does that mean the next roll is less likely to be a 6 for a fair die? Explain.


# Part C — Sampling With and Without Replacement

Recall:

- With replacement: an item can be selected again.
- Without replacement: a selected item is removed from the available pool.
- A simple random sample is drawn randomly without replacement.


In [ ]:
students = np.array([
    "Ava", "Ben", "Carlos", "Dina", "Eli",
    "Fatima", "Grace", "Hector", "Ivy", "Jamal"
])

students


### Activity 3 — Compare the two sampling methods


In [ ]:
# select 6 students WITH replacement.
sample_with = rng.choice(students, size=6, replace=True)

# select 6 students WITHOUT replacement.
sample_without = rng.choice(students, size=6, replace=False)

print("With replacement:   ", sample_with)
print("Without replacement:", sample_without)


### Quick Check 3

1. In which sample can the same student appear more than once?
2. If you wanted to select six different students for a committee, which method would you use?
3. If you wanted to model repeated die rolls, which method would you use?



# Part D — Population, Parameter, Sample, and Statistic

We will create a synthetic population of 5,000 flight departure delays.

For this activity, treat all 5,000 values as the population.

The population median is a parameter. A median calculated from a random sample is a statistic.


In [ ]:
# synthetic population of flight delays (minutes)
# most flights have modest delays, but a few have long delays.
flight_delays = np.round(rng.gamma(shape=2.0, scale=12.0, size=5000) - 8, 1)

population = pd.DataFrame({
    "Delay": flight_delays
})

population.head()


In [ ]:
population_median = population["Delay"].median()
population_mean = population["Delay"].mean()

print("Population size:", len(population))
print("Population median delay:", round(population_median, 2))
print("Population mean delay:", round(population_mean, 2))


In [ ]:
plt.hist(population["Delay"], bins=30)
plt.xlabel("Departure delay (minutes)")
plt.ylabel("Number of flights")
plt.title("Population Distribution of Flight Delays")
plt.show()


### Activity 4 — Take one random sample

Take a simple random sample of 50 flights and calculate the sample median.


In [ ]:
# draw 50 rows WITHOUT replacement.
sample_50 = population.sample(n=50, replace=False)

# calculate the sample median.
sample_median_50 = sample_50["Delay"].median()

print("Population median:", round(population_median, 2))
print("Sample median:", round(sample_median_50, 2))


### Quick Check 4

Identify each quantity.

1. Median delay of all 5,000 flights: parameter or statistic?
2. Median delay of the 50 sampled flights: parameter or statistic?
3. Re-run the sample cell. Why does the sample median change while the population median stays fixed?


# Part E — Why Simulate a Statistic?

One random sample gives one statistic. Another random sample usually gives a different statistic.

To understand this sampling variability, we repeat the sampling process many times and store the statistic from every sample.

The resulting values form an empirical distribution of the statistic.


### Activity 5 — Write a function that returns one sample median

Complete the function.


In [ ]:
def random_sample_median(sample_size):
    # draw a random sample of the requested size without replacement.
    sample = population.sample(n=sample_size, replace=False)

    # return the median of the Delay column.
    return sample["Delay"].median()

# test the function
random_sample_median(50)


### Activity 6 — Repeat the sampling process 2,000 times


In [ ]:
repetitions = 2_000
sample_medians = []

# repeat the random sampling process.
for i in range(repetitions):
    med = random_sample_median(50)
    sample_medians.append(med)

sample_medians = np.array(sample_medians)

print("Number of simulated statistics:", len(sample_medians))
print("First 10 sample medians:", sample_medians[:10])


In [ ]:
plt.hist(sample_medians, bins=25)
plt.axvline(population_median, linestyle="--",
            label=f"Population median = {population_median:.1f}")
plt.xlabel("Sample median delay")
plt.ylabel("Frequency")
plt.title ("Empirical Distribution of the Sample Median 2,000 samples of size 50")
plt.legend()
plt.show()


# Quick Check 5

1. What does one value in sample medians`represent?
2. Why are the 2,000 values not all identical?
3. What does the vertical dashed line represent?
4. Is the histogram above the distribution of individual flight delays, or the distribution of a statistic?

# Part F — How Does Sample Size Affect Variability?

Now compare the empirical distribution of the sample median for:

- samples of size 10
- samples of size 50
- samples of size 200

Use 2,000 repetitions for each.


In [ ]:
def simulate_sample_medians(sample_size, repetitions=2000):
    medians = []

    for i in range(repetitions):
        medians.append(random_sample_median(sample_size))

    return np.array(medians)

medians_10 = simulate_sample_medians(10)
medians_50 = simulate_sample_medians(50)

# simulate sample medians for sample size 200.
medians_200 = simulate_sample_medians(200)


In [ ]:
# use the same horizontal range so the spreads are easy to compare.
all_medians = np.concatenate([medians_10, medians_50, medians_200])
x_min, x_max = all_medians.min(), all_medians.max()

for sample_size, values in [(10, medians_10), (50, medians_50), (200, medians_200)]:
    plt.figure(figsize=(7, 3))
    plt.hist(values, bins=25, range=(x_min, x_max))
    plt.axvline(population_median, linestyle="--",
                label="Population median")
    plt.xlim(x_min, x_max)
    plt.xlabel("Sample median")
    plt.ylabel("Frequency")
    plt.title(f"Sample Median Distribution — n = {sample_size}")
    plt.legend()
    plt.show()


In [ ]:
#sampling distribution of median
summary = pd.DataFrame({
    "Sample size": [10, 50, 200],
    "Mean of simulated medians": [
        medians_10.mean(),
        medians_50.mean(),
        medians_200.mean()
    ],
    "SD of simulated medians": [
        medians_10.std(),
        medians_50.std(),
        medians_200.std()
    ]
})

summary


#Quick Check 6

1. Which sample size produces the widest distribution of sample medians?
2. Which produces the narrowest distribution?
3. What happens to the standard deviation of the simulated medians as sample size increases?
4. In your own words, what does this tell you about larger random samples?


# Part G — Challenge: Sampling from a Categorical Distribution

A flower population has the following proportions:

- Red: 25%
- Pink: 50%
- White: 25%

We will simulate a random sample of 300 flowers.


In [ ]:
colors = np.array(["Red", "Pink", "White"])
population_probabilities = np.array([0.25, 0.50, 0.25])

# simulate 300 flowers using rng.choice.
flower_sample = rng.choice(
    colors,
    size=300,
    replace=True,
    p=population_probabilities
)

sample_proportions = np.array([
    (flower_sample == color).mean()
    for color in colors
])

flower_results = pd.DataFrame({
    "Color": colors,
    "Population proportion": population_probabilities,
    "Sample proportion": sample_proportions
})

flower_results


# Challenge Question

Run the flower simulation several times.

Why do the sample proportions change? Why are they usually reasonably close to 0.25, 0.50, and 0.25?

---

## Optional Extension — Track Convergence Directly

The code below tracks the cumulative proportion of 4s as the number of die rolls grows.


In [ ]:
n_rolls = 10_000
rolls = rng.choice(faces, size=n_rolls)

is_four = (rolls == 4)
running_proportion = np.cumsum(is_four) / np.arange(1, n_rolls + 1)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, n_rolls + 1), running_proportion)
plt.axhline(1/6, linestyle="--", label="Theoretical P(4) = 1/6")
plt.xlabel("Number of rolls")
plt.ylabel("Cumulative proportion of 4s")
plt.title("Law of Averages in Action")
plt.legend()
plt.show()


# Extension Reflection

Describe what happens to the cumulative proportion as the number of rolls becomes large. Does it become perfectly constant?

